In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:,.4f}".format)


In [2]:
def exclude_financial_services(df):
    """
    Exclude financial services sectors
    """
    financial_sectors = [
        'Banks',
        'Financial Services',
        'NBFC',
        'Insurance'
    ]
    return df[~df['sector'].isin(financial_sectors)]


def handle_missing_values(df):
    """
    Remove rows with NaN or infinite values
    """
    df_clean = df.dropna()

    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    mask_inf = df_clean[numeric_cols].applymap(
        lambda x: np.isinf(x) if isinstance(x, (int, float)) else False
    ).any(axis=1)

    return df_clean[~mask_inf]


In [11]:
def calculate_rolling_features(df, window=3):
    """
    Calculate rolling 3-year financial features per company
    """
    features_df = df.copy()
    features_df = features_df.sort_values(['company_id', 'financial_year_end'])

    numeric_cols = [
        'operating_cf',
        'investing_cf',
        'financing_cf',
        'revenue',
        'profit_after_tax',
        'total_assets',
        'total_liabilities',
        'equity'
    ]

    rolling_features = []

    for company_id, company_data in features_df.groupby('company_id'):
        company_data = company_data.sort_values('financial_year_end')

        if len(company_data) < window + 1:
            continue

        for i in range(window, len(company_data)):
            current_row = company_data.iloc[i]
            lookback = company_data.iloc[i - window:i]

            # Skip incomplete windows
            if lookback[numeric_cols].isnull().any().any():
                continue

            row = {
                'company_id': company_id,
                'financial_year_end': current_row['financial_year_end'],
                'label': current_row.get('label', np.nan)
            }

            # Rolling stats
            for col in numeric_cols:
                row[f'{col}_3yr_mean'] = lookback[col].mean()
                row[f'{col}_3yr_std'] = lookback[col].std()

           
            # Financing dependency & OCF stress
            
            
            EPSILON = 1e-6
            
            operating_cf_mean = lookback['operating_cf'].mean()
            financing_cf_mean = lookback['financing_cf'].mean()
            
            # Financing dependency ratio
            row['financing_dependency_ratio'] = (
                financing_cf_mean / abs(operating_cf_mean + EPSILON)
            )
            
            # % of years with negative OCF in lookback window
            row['pct_years_ocf_negative'] = (
                (lookback['operating_cf'] < 0).mean()
            )


            # Growth
            if len(lookback) >= 2:
                row['revenue_3yr_growth'] = (
                    lookback['revenue'].iloc[-1] - lookback['revenue'].iloc[0]
                ) / lookback['revenue'].iloc[0]

                row['profit_3yr_growth'] = (
                    lookback['profit_after_tax'].iloc[-1] - lookback['profit_after_tax'].iloc[0]
                ) / lookback['profit_after_tax'].iloc[0]

            rolling_features.append(row)

    return pd.DataFrame(rolling_features)


In [12]:
canonical_df = pd.read_csv("canonical_financials.csv")
labeled_df = pd.read_csv("labeled_financials.csv")

df = canonical_df.merge(
    labeled_df[['company_id', 'financial_year_end', 'label']],
    on=['company_id', 'financial_year_end'],
    how='inner'
)

print(f"Initial merged dataset: {len(df)} rows")
df.head()


Initial merged dataset: 815 rows


,company_id,financial_year_end,operating_cf,investing_cf,financing_cf,net_cf,total_assets,total_liabilities,equity,revenue,profit_after_tax,sector,face_value,book_value,roe_trailing,roce_trailing,label
0,ABB,2018,"2,198.0000","-3,192.0000","1,589.0000",596.0000,"2,416.0000","2,416.0000","1,693.0000",3298,401,Capital Goods,10,1657,34.9000,46.0000,0
1,ABB,2019,"2,591.0000","-3,050.0000",38.0000,-421.0000,"2,941.0000","2,941.0000","2,008.0000",3679,450,Capital Goods,10,1657,34.9000,46.0000,0
2,ABB,2020,"5,437.0000","-5,643.0000","1,250.0000","1,045.0000","3,547.0000","3,547.0000","2,431.0000",4093,593,Capital Goods,10,1657,34.9000,46.0000,0
3,ABB,2021,"3,784.0000","-4,009.0000",-745.0000,-969.0000,"3,840.0000","3,840.0000","2,602.0000",4310,691,Capital Goods,10,1657,34.9000,46.0000,0
4,ABB,2022,"4,097.0000","-3,936.0000",-235.0000,-75.0000,"4,224.0000","4,224.0000","2,820.0000",4913,799,Capital Goods,10,1657,34.9000,46.0000,0


In [13]:
df_filtered = exclude_financial_services(df)

print(f"After excluding financial services: {len(df_filtered)} rows")

print("\nSector distribution:")
df_filtered['sector'].value_counts()


After excluding financial services: 626 rows

Sector distribution:


sector
Metals & Mining                   72
Fast Moving Consumer Goods        71
Automobile and Auto Components    62
Capital Goods                     58
Power                             57
Healthcare                        54
Information Technology            42
Oil Gas & Consumable Fuels        36
Diversified / Others              36
Construction Materials            33
Retail                            30
Transportation / Logistics        23
Consumer Durables                 18
Construction Real Estate          16
Telecommunication                  9
Chemicals                          9
Name: count, dtype: int64

In [14]:
features_df = calculate_rolling_features(df_filtered, window=3)

print(f"Rows after rolling feature calculation: {len(features_df)}")
features_df.head()

Rows after rolling feature calculation: 401


,company_id,financial_year_end,label,operating_cf_3yr_mean,operating_cf_3yr_std,investing_cf_3yr_mean,investing_cf_3yr_std,financing_cf_3yr_mean,financing_cf_3yr_std,revenue_3yr_mean,revenue_3yr_std,profit_after_tax_3yr_mean,profit_after_tax_3yr_std,total_assets_3yr_mean,total_assets_3yr_std,total_liabilities_3yr_mean,total_liabilities_3yr_std,equity_3yr_mean,equity_3yr_std,financing_dependency_ratio,pct_years_ocf_negative,revenue_3yr_growth,profit_3yr_growth
0,ABB,2021,0,"3,408.6667","1,767.5447","-3,961.6667","1,457.8074",959.0000,815.4208,"3,690.0000",397.6141,481.3333,99.7614,"2,968.0000",565.9832,"2,968.0000",565.9832,"2,044.0000",370.3147,0.2813,0.0000,0.2411,0.4788
1,ABB,2022,0,"3,937.3333","1,429.1824","-4,234.0000","1,311.0610",181.0000,"1,005.1582","4,027.3333",320.5844,578.0000,121.1982,"3,442.6667",458.4914,"3,442.6667",458.4914,"2,347.0000",305.7793,0.0460,0.0000,0.1715,0.5356
2,ABB,2023,0,"4,439.3333",878.0640,"-4,529.3333",965.1540,90.0000,"1,036.4483","4,438.6667",424.8721,694.3333,103.0404,"3,870.3333",339.5178,"3,870.3333",339.5178,"2,617.6667",194.9726,0.0203,0.0000,0.2003,0.3474
3,ABB,2024,0,"3,886.0000",182.7649,"-4,214.6667",421.0301,-19.0000,854.7210,"4,857.3333",521.7321,813.0000,129.5685,"4,206.6667",358.3146,"4,206.6667",358.3146,"2,870.0000",296.1824,-0.0049,0.0000,0.2411,0.3734
4,ADANIENSOL,2021,0,"3,408.6667","1,767.5447","-3,961.6667","1,457.8074",959.0000,815.4208,"7,555.0000","3,742.2682",802.6667,303.7636,"29,844.6667","11,466.3578","29,844.6667","11,466.3578","7,533.0000","1,298.4283",0.2813,0.0000,1.8945,-0.3823


In [15]:
features_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 401 entries, 0 to 400
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   company_id                  401 non-null    object 
 1   financial_year_end          401 non-null    int64  
 2   label                       401 non-null    int64  
 3   operating_cf_3yr_mean       401 non-null    float64
 4   operating_cf_3yr_std        401 non-null    float64
 5   investing_cf_3yr_mean       401 non-null    float64
 6   investing_cf_3yr_std        401 non-null    float64
 7   financing_cf_3yr_mean       401 non-null    float64
 8   financing_cf_3yr_std        401 non-null    float64
 9   revenue_3yr_mean            401 non-null    float64
 10  revenue_3yr_std             401 non-null    float64
 11  profit_after_tax_3yr_mean   401 non-null    float64
 12  profit_after_tax_3yr_std    401 non-null    float64
 13  total_assets_3yr_mean       401 non

In [16]:
print("=== Final Dataset Summary ===")
print(f"Total rows: {len(features_df)}")
print(f"Companies: {features_df['company_id'].nunique()}")

risk_rate = features_df['label'].mean() * 100
print(f"Risk rate: {risk_rate:.1f}%")
print(f"Healthy (0): {(features_df['label'] == 0).sum()}")
print(f"Risky (1): {(features_df['label'] == 1).sum()}")


=== Final Dataset Summary ===
Total rows: 401
Companies: 71
Risk rate: 8.7%
Healthy (0): 366
Risky (1): 35


In [17]:
output_path = "modeling_dataset_v1.csv"
features_df.to_csv(output_path, index=False)

print(f"Modeling dataset saved to: {output_path}")


Modeling dataset saved to: modeling_dataset_v1.csv
